# リップシンク テスト (SadTalker on Colab)
静止画 `singer.png` + ボーカル `01Voice.wav` (9.3秒) から、口が音声に合う動画を作ります。

**進め方**: 上から順にセルを実行 (▶ボタン) してください。赤いエラーが出たら、その全文をマネージャーに送ってください。直したノートブックを返します。
まず メニュー「ランタイム」→「ランタイムのタイプを変更」→ GPU (T4) を選んで保存してから始めてください。

In [ ]:
# 1) GPU確認 (T4などと表示されればOK)
!nvidia-smi -L

In [ ]:
# 2) SadTalker を取得
%cd /content
!git clone -q https://github.com/OpenTalker/SadTalker.git
%cd /content/SadTalker
!ls

In [ ]:
# 3) 依存パッケージ (小分けインストール — 1個失敗しても他は入る)
!pip install -q numpy==1.26.4 face_alignment==1.3.5 facexlib==0.3.0
!pip install -q kornia
!pip install -q librosa==0.10.2.post1 resampy pydub
!pip install -q numba resampy face_alignment==1.3.5 facexlib==0.3.0
!pip install -q scipy scikit-image yacs pyyaml joblib imageio imageio-ffmpeg tqdm av safetensors
# kornia の import パスを SadTalker 側に合わせておく
!sed -i 's/from kornia.geometry import warp_affine/from kornia.geometry.transform import warp_affine/' /content/SadTalker/src/face3d/models/networks.py
# 全部入ったか自動チェック
import importlib
need = ["kornia","face_alignment","facexlib","librosa","scipy","skimage","yacs","imageio","pydub","resampy","numba","tqdm","yaml","joblib","cv2","av","safetensors"]
missing = []
for m in need:
    try: importlib.import_module(m)
    except Exception as e:
        missing.append(m)
        print("NG", m, "->", type(e).__name__, str(e)[:300])
print("MISSING:", missing if missing else "none - all OK")

In [ ]:
# 4) モデル (チェックポイント) をダウンロード — 約2GB、数分かかります
!bash scripts/download_models.sh
!ls checkpoints

In [ ]:
# 5) 素材をアップロード: 歌い手の画像 (png/jpg) と 音声 (mp3/wav) の2つを選んでください
#    ファイル名は何でもOK — 中身で見分けて singer.png / 01Voice.wav に整えます
from google.colab import files
up = files.upload()
import glob, os, shutil
def newest(patterns):
    hits = []
    for pat in patterns: hits += glob.glob(pat)
    return max(hits, key=os.path.getmtime) if hits else None
img = newest(['*.png', '*.jpg', '*.jpeg'])
aud = newest(['*.mp3', '*.wav'])
print('image file:', img)
print('audio file:', aud)
shutil.copy(img, '/content/SadTalker/singer.png')
if aud.lower().endswith('.mp3'):
    from pydub import AudioSegment
    a = AudioSegment.from_mp3(aud).set_frame_rate(16000).set_channels(1)
    a.export('/content/SadTalker/01Voice.wav', format='wav')
else:
    shutil.copy(aud, '/content/SadTalker/01Voice.wav')
print('singer.png', os.path.getsize('/content/SadTalker/singer.png'))
print('01Voice.wav', os.path.getsize('/content/SadTalker/01Voice.wav'))


In [ ]:
# 6) 生成 (9秒クリップ: T4で数分が目安)
%cd /content/SadTalker
# gfpgan は enhancer 用で今回は未使用。ただし SadTalker が起動時に無条件 import するため、
# 読めなくても止まらないよう任意化パッチを当てる (冪等)
p = '/content/SadTalker/src/utils/face_enhancer.py'
s = open(p).read()
if 'GFPGANer = None' not in s:
    s = s.replace('from gfpgan import GFPGANer',
                  'try:\n    from gfpgan import GFPGANer\nexcept Exception:\n    GFPGANer = None')
    open(p, 'w').write(s)
print('face_enhancer patch: done')
# 生成前に素材の場所を最終確認 (False なら セル5 をやり直してください)
import os
print('singer.png:', os.path.isfile('/content/SadTalker/singer.png'),
      ' 01Voice.wav:', os.path.isfile('/content/SadTalker/01Voice.wav'))
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python inference.py --driven_audio /content/SadTalker/01Voice.wav --source_image /content/SadTalker/singer.png --result_dir ./results --still --preprocess full
!ls -la results

In [ ]:
# 7) 結果を表示してダウンロード
import glob
from IPython.display import Video, display
vids = sorted(glob.glob('results/*.mp4'))
print(vids)
display(Video(vids[-1], width=640, embed=True))
files.download(vids[-1])

## 終わったら
ダウンロードした mp4 をマネージャーに送ってください。フレームチェックして、必要なら連結や音声の仕上げをこちらでやります。
うまく行かない場合は、エラーが出たセルの赤い文字を全部コピーして送ってください。